# Agentic AI with Python SDKs

This notebook introduces core agentic AI concepts using three major Python SDKs:

| SDK | Provider | Install |
|---|---|---|
| `anthropic` | Anthropic (Claude) | `pip install anthropic` |
| `openai` | OpenAI (GPT / o-series) | `pip install openai` |
| `langchain` + `langgraph` | Provider-agnostic orchestration | `pip install langchain langgraph langchain-anthropic langchain-openai` |

Each section focuses on one concept and **contrasts behavior with and without** that configuration so differences are concrete.

---

**Sections**
1. Setup & Installation
2. Model Choice
3. API Types — Chat Completions vs Responses API vs LangChain invoke
4. Message Types — system / user / assistant
5. Tools
6. Memory
7. Streaming
8. Structured Output

---
## 1. Setup & Installation

In [5]:
%pip install -q anthropic openai langchain langgraph langchain-anthropic langchain-openai pydantic
%pip install -q python-dotenv

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [6]:
from dotenv import load_dotenv

# Load your API keys from a .env file
load_dotenv('.env')

True

In [7]:
# Initialize clients — reused throughout the notebook
import anthropic
from openai import OpenAI

anthropic_client = anthropic.Anthropic()
openai_client    = OpenAI()

print("Clients ready.")

Clients ready.


---
## 2. Model Choice

The model controls capability, speed, and cost. Choosing the right model for the task is the first agentic decision.

### Anthropic Claude model tiers

| Model ID | Tier | Best for |
|---|---|---|
| `claude-opus-4-7` | Frontier | Complex reasoning, long documents, research |
| `claude-sonnet-4-6` | Balanced | Most production agentic tasks |
| `claude-haiku-4-5-20251001` | Fast / cheap | High-volume classification, extraction |

### OpenAI model tiers

| Model ID | Tier | Best for |
|---|---|---|
| `gpt-5.5` | Frontier | Multimodal, tool use, complex agents |
| `gpt-5.4` | Balanced | Cost-effective agents |
| `gpt-5.4-mini` | Fast reasoning | Lightweight reasoning tasks |

In [8]:
# Anthropic: swapping models changes only the `model` argument
def ask_claude(question: str, model: str) -> str:
    response = anthropic_client.messages.create(
        model=model,
        max_tokens=256,
        messages=[{"role": "user", "content": question}],
    )
    return response.content[0].text

question = "Name three key differences between DNA and RNA in one sentence each."

print("=== Haiku (fast) ===")
print(ask_claude(question, model="claude-haiku-4-5-20251001"))

print("\n=== Sonnet (balanced) ===")
print(ask_claude(question, model="claude-sonnet-4-6"))

=== Haiku (fast) ===
# Three Key Differences Between DNA and RNA

1. **Sugar composition**: DNA contains deoxyribose sugar, while RNA contains ribose sugar (which has an extra hydroxyl group).

2. **Bases**: DNA uses thymine as a nitrogenous base, whereas RNA uses uracil instead.

3. **Structure**: DNA is typically double-stranded, while RNA is usually single-stranded.

=== Sonnet (balanced) ===
Here are three key differences between DNA and RNA:

1. **Sugar**: DNA contains deoxyribose sugar (lacking a hydroxyl group at the 2' carbon), while RNA contains ribose sugar (with a hydroxyl group at the 2' carbon).

2. **Bases**: DNA uses the nitrogenous base thymine, whereas RNA replaces thymine with uracil, which lacks a methyl group.

3. **Strand structure**: DNA typically exists as a double-stranded helix, while RNA is generally single-stranded, allowing it to fold into various functional shapes.


In [11]:
# OpenAI: same pattern — only the `model` argument changes
def ask_gpt(question: str, model: str) -> str:
    response = openai_client.chat.completions.create(
        model=model,
        max_completion_tokens=256,
        messages=[{"role": "user", "content": question}],
    )
    return response.choices[0].message.content

print("=== GPT-5.4 mini (fast) ===")
print(ask_gpt(question, model="gpt-5.4-mini"))

print("\n=== GPT-5.4 (balanced) ===")
print(ask_gpt(question, model="gpt-5.4"))

=== GPT-5.4 mini (fast) ===
1. DNA contains the sugar **deoxyribose**, while RNA contains **ribose**.  
2. DNA usually has the base **thymine**, whereas RNA uses **uracil** instead.  
3. DNA is typically **double-stranded** and stores genetic information, while RNA is usually **single-stranded** and helps carry out gene expression.

=== GPT-5.4 (balanced) ===
1. DNA contains the sugar deoxyribose, while RNA contains ribose.  
2. DNA uses the base thymine, while RNA uses uracil instead.  
3. DNA is usually double-stranded, while RNA is usually single-stranded.


In [13]:
# LangChain: model is set at init time — the rest of the chain stays identical
from langchain_anthropic import ChatAnthropic
from langchain_openai import ChatOpenAI

claude_haiku  = ChatAnthropic(model="claude-haiku-4-5-20251001", max_tokens=256)
gpt54_mini    = ChatOpenAI(model="gpt-5.4-mini",                  max_tokens=256)

# All three use the same .invoke() interface
for name, model in [("Haiku", claude_haiku), ("GPT-5.4-mini", gpt54_mini)]:
    result = model.invoke(question)
    print(f"\n=== {name} ===")
    print(result.content)


=== Haiku ===
# Three Key Differences Between DNA and RNA

1. **Sugar component**: DNA contains deoxyribose sugar, while RNA contains ribose sugar (which has an extra hydroxyl group).

2. **Nitrogenous bases**: DNA uses thymine as a base, while RNA uses uracil instead.

3. **Structure**: DNA is typically double-stranded, while RNA is typically single-stranded.

=== GPT-5.4-mini ===
1. **DNA** contains the sugar **deoxyribose**, while **RNA** contains **ribose**.  
2. **DNA** usually has the base **thymine (T)**, while **RNA** uses **uracil (U)** instead.  
3. **DNA** is typically **double-stranded**, while **RNA** is usually **single-stranded**.


---
## 3. API Types

Each provider exposes different API surfaces. OpenAI offers two distinct APIs for different use cases.

| API | Method | Primary use |
|---|---|---|
| Anthropic **Messages API** | `client.messages.create()` | All Claude interactions |
| OpenAI **Chat Completions API** | `client.chat.completions.create()` | Standard multi-turn chat |
| OpenAI **Responses API** | `client.responses.create()` | Stateful agents with built-in tools |
| LangChain **invoke** | `model.invoke()` / `chain.invoke()` | Provider-agnostic pipeline |

In [ ]:
# ── Anthropic Messages API ──────────────────────────────────────────────────
response = anthropic_client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=128,
    messages=[{"role": "user", "content": "What is PCR used for?"}],
)
print("Anthropic Messages API:")
print(response.content[0].text)

In [ ]:
# ── OpenAI Chat Completions API ─────────────────────────────────────────────
# The classic API: stateless, you manage the full message history yourself.
response = openai_client.chat.completions.create(
    model="gpt-5.4-mini",
    max_tokens=128,
    messages=[{"role": "user", "content": "What is PCR used for?"}],
)
print("OpenAI Chat Completions API:")
print(response.choices[0].message.content)

In [ ]:
# ── OpenAI Responses API ────────────────────────────────────────────────────
# Newer stateful API designed for agents. OpenAI manages conversation state
# server-side via `previous_response_id`. Also has built-in tools:
# web_search_preview, code_interpreter, file_search.

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input="What is PCR used for?",
    # instructions="You are a molecular biology expert."  # optional system prompt
)
print("OpenAI Responses API:")
print(response.output_text)
print(f"\nResponse ID (use as previous_response_id to continue): {response.id}")

In [ ]:
# Responses API: continuing a conversation using previous_response_id
follow_up = openai_client.responses.create(
    model="gpt-5.4-mini",
    input="What are the three main steps of the process you just described?",
    previous_response_id=response.id,   # server retains prior context
)
print("Follow-up (Responses API with state):")
print(follow_up.output_text)

---
## 4. Message Types

Conversations are composed of messages with different **roles**. The role tells the model who is speaking.

- `system` — instructions to the model, not part of the dialogue. Sets persona, constraints, output format, tone. The model treats this as "how I should behave" rather than "something someone said to me."
- `user` — input from the human turn. The model treats this as "what I'm being asked."
- `assistant` — prior model outputs injected back into the message list. Tells the model "this is what I already said," so it stays consistent and doesn't contradict itself.

System prompt carries greater influence on an agent's responses than other message types. The system prompt shapes every subsequent response in the conversation. A system prompt saying "answer in one sentence" constrains the model far more reliably than repeating that instruction as a user message. The user/assistant turns provide the conversational context the model reasons over.

| Role | Anthropic | OpenAI Chat | LangChain |
|---|---|---|---|
| System prompt | `system` param (top-level) | `{"role": "system", ...}` | `SystemMessage(...)` |
| Human turn | `{"role": "user", ...}` | `{"role": "user", ...}` | `HumanMessage(...)` |
| Model turn | `{"role": "assistant", ...}` | `{"role": "assistant", ...}` | `AIMessage(...)` |

**The system prompt is the most powerful lever** — it sets the model's persona, constraints, and output style for the entire conversation.

In [14]:
user_question = "Explain what a p-value is."

# ── Without a system prompt ─────────────────────────────────────────────────
no_system = anthropic_client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=150,
    messages=[{"role": "user", "content": user_question}],
)
print("WITHOUT system prompt:")
print(no_system.content[0].text)

WITHOUT system prompt:
# What is a P-Value?

## The Core Idea

A p-value is the **probability of observing results at least as extreme as your data, *assuming the null hypothesis is true*.**

---

## Breaking That Down

**The null hypothesis** is usually the "nothing interesting is happening" claim — for example:
- "This drug has no effect"
- "These two groups are the same"

**The p-value asks:** *If the null hypothesis were really true, how likely would I be to get data like this (or more extreme) just by random chance?*

---

## A Simple Example

You flip a coin 10 times and get 9 heads


In [16]:
# ── With a system prompt ────────────────────────────────────────────────────
with_system = anthropic_client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=150,
    system="You are a statistics tutor explaining concepts to a first-year biology student. "
           "Use plain language, one concrete example, and avoid formulas.",
    messages=[{"role": "user", "content": user_question}],
)
print("WITH system prompt (biology student persona):")
print(with_system.content[0].text)

WITH system prompt (biology student persona):
# What is a P-Value?

## The Simple Idea

A p-value tells you **how surprising your results would be if nothing interesting was actually happening.**

Think of it like this: it measures the probability of getting your results (or something even more extreme) purely by chance.

---

## A Concrete Example

Say you want to know if a new fertilizer makes plants grow taller. You treat 20 plants and leave 20 plants untreated. The treated plants end up taller on average.

But here's the key question: **could that just be random luck?** Maybe you accidentally picked naturally taller plants for the treatment group.

The p-value asks: *


In [30]:
# ── Multi-turn conversation (Anthropic) ─────────────────────────────────────
# Assistant turns are injected into `messages` to maintain history.
messages = [
    {"role": "user",      "content": "My experiment has p=0.03. Is it significant?"},
    {"role": "assistant", "content": "Yes — p=0.03 is below the common α=0.05 threshold, "
                                     "so you would reject the null hypothesis."},
    {"role": "user",      "content": "What if I run 20 tests and one gives p=0.03?"},
]

response = anthropic_client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=200,
    system="You are a statistics tutor for biology students.",
    messages=[
        {
            "role": "user",
            "content": "My experiment has p=0.03. Is it significant?"
        },
        {
            "role": "assistant",
            "content": "Yes — p=0.03 is below the common α=0.05 threshold, so you would reject the null hypothesis."
        },
        {
            "role": "user",
            "content": "What if I run 20 tests and one gives p=0.03?"
        },
    ],
)
print("Multi-turn conversation — turn 3:")
print(response.content[0].text)

Multi-turn conversation — turn 3:
This is a really important question! This is actually a problem called **multiple comparisons**.

## The Issue

If you run **20 tests** at α = 0.05, you would expect **1 false positive** just by chance, even if nothing is real.

- Probability of a false positive per test = 0.05
- Expected false positives in 20 tests = 20 × 0.05 = **1**

So your p = 0.03 result could easily be that **chance false positive**.

## What You Should Do

Consider a **correction for multiple comparisons**, such as:

| Method | How it works |
|--------|-------------|
| **Bonferroni** | Divide α by number of tests (0.05/20 = **0.0025** required) |
| **FDR (Benjamini-Hochberg


In [ ]:
# ── LangChain message types ─────────────────────────────────────────────────
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

model = ChatAnthropic(model="claude-sonnet-4-6", max_tokens=150)

messages = [
    SystemMessage(content="You are a concise scientific writing assistant."),
    HumanMessage(content="Write a one-sentence definition of epigenetics."),
]

response = model.invoke(messages)
print("LangChain — AIMessage response:")
print(response.content)
print(f"\nResponse type: {type(response).__name__}")

In [ ]:
# ── LangChain multi-turn using message list ─────────────────────────────────
history = [
    SystemMessage(content="You are a concise scientific writing assistant."),
    HumanMessage(content="Write a one-sentence definition of epigenetics."),
    AIMessage(content=response.content),  # inject prior assistant turn
    HumanMessage(content="Now give me one example of an epigenetic modification."),
]

follow_up = model.invoke(history)
print("LangChain multi-turn — second turn:")
print(follow_up.content)

In [35]:
# The system prompt defines what the agent is allowed to answer.                              
# Even when the user tries to stray, the operator-level constraint holds.

genomics_system = (
    "You are a genomics assistant. "
    "Answer only questions about genes, genomes, sequencing, or molecular biology. "
    "For any other topic, politely decline and redirect to genomics."
)

questions = [
    "What does high GC content mean for PCR primer design?",  # in scope
    "Can you recommend a good restaurant in Boston?",          # out of scope
    "What is the capital of France?",                          # out of scope
]

for q in questions:
    response = anthropic_client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=100,
        system=genomics_system,
        messages=[{"role": "user", "content": q}],
    )
    print(f"Q: {q}")
    print(f"A: {response.content[0].text}\n")

Q: What does high GC content mean for PCR primer design?
A: ## GC Content in PCR Primer Design

High GC content has several important implications for primer design:

### Why GC Content Matters
- G-C base pairs form **3 hydrogen bonds** (vs. 2 for A-T pairs)
- This makes GC-rich sequences more thermally stable
- Directly affects melting temperature (Tm)

### Optimal Range
- **Ideal GC content: 40–60

Q: Can you recommend a good restaurant in Boston?
A: I appreciate you asking, but restaurant recommendations are outside my area of expertise! I focus specifically on **genomics, genes, sequencing, and molecular biology** topics.

However, if you have questions in those areas, I'd love to help! For example, I could help with:

- 🧬 **Gene function or regulation**
- 🔬 **Sequencing technologies** (NGS, long-read, etc.)
- 

Q: What is the capital of France?
A: That's a geography question, which is outside my area of expertise! I'm specialized in **genomics, molecular biology, and sequencing to

In [33]:
conflicting.content[0].text

'```json\n{\n  "definition": "DNA, or deoxyribonucleic acid, is the hereditary material found in the cells of all living organisms that carries the genetic instructions for the development, functioning, growth, and reproduction of life.",\n  "key_facts": [\n    "DNA is a double-stranded molecule twisted into a double helix structure.",\n    "It is composed of four nucleotide bases: adenine (A), thymine (T), guanine (G), and cytosine (C).",\n    "Base pairing rules dictate that A pairs with T, and G pairs with C.",\n    "DNA is found primarily in the cell nucleus, as well as in mitochondria.",\n    "Segments of DNA called genes encode instructions for building proteins.",\n    "DNA replicates itself during cell division to pass genetic information to new cells.",\n    "The human genome contains'

In [23]:
r1_user.content[0].text

'# What is DNA?\n\n• **Full name**: Deoxyribonucleic acid\n\n• **Definition**: A molecule that carries genetic instructions for life in all living organisms\n\n• **Location**: Found mainly in the nucleus of cells (also in mitochondria and chloroplasts)\n\n• **Structure**: Double helix shape made of two intertwined strands\n\n• **Building blocks**: Made of nucleotides, which contain:\n  - A sugar (deoxyribose)\n  - A phosphate group\n  - A nitrogenous base\n\n• **Base pairs**: Four types of bases pair together:\n  - Adenine (A) pairs with Thymine (T)\n  - Guanine (G) pairs with Cytosine (C)\n\n• **Function**: Stores and transmits genetic information from one generation to the next\n\n• **Genes**: Segments of DNA that code'

In [21]:
history_sys

[{'role': 'user', 'content': 'Explain what DNA is.'},
 {'role': 'assistant',
  'content': '# DNA Overview\n\n• **Full name**: Deoxyribonucleic acid\n\n• **What it is**: A molecule that carries genetic instructions for all living organisms\n\n• **Location**: Found primarily in the nucleus of cells (also in mitochondria and chloroplasts)\n\n• **Structure**: \n  - Double helix shape (twisted ladder)\n  - Made up of two strands wound together\n  - Composed of four chemical bases: adenine (A), thymine (T), guanine (G), and cytosine (C)\n\n• **Basic building blocks**: \n  - Sugar (deoxyribose)\n  - Phosphate groups\n  - Nitrogenous bases\n\n• **Function**: Stores and transmits genetic information from one generation to the next\n\n• **Key features**:\n  - Bases pair in specific ways (A with T, G with C)\n  '},
 {'role': 'user', 'content': 'Now explain RNA.'}]

---
## 5. Tools

Tools (also called function calling) let the model request external actions — database queries, web searches, API calls, calculations — and receive the results before answering.

**Without tools**: the model answers from training knowledge only (may hallucinate facts).
**With tools**: the model can fetch live, precise, or private data.

The tool loop:
```
User message → Model → tool_use request → Your code runs the tool
    → tool_result back to model → Model → final answer
```

In [ ]:
# ── Without tools (model guesses from training data) ────────────────────────
response_no_tool = anthropic_client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=100,
    messages=[{"role": "user", "content": "What is the GC content of the BRCA1 gene?"}],
)
print("WITHOUT tools:")
print(response_no_tool.content[0].text)

In [ ]:
import json

# ── With tools: Anthropic tool definition + execution loop ──────────────────
tools = [
    {
        "name": "get_gene_info",
        "description": "Retrieve basic information about a gene by symbol, "
                        "including GC content, chromosome, and length.",
        "input_schema": {
            "type": "object",
            "properties": {
                "gene_symbol": {
                    "type": "string",
                    "description": "HGNC gene symbol, e.g. BRCA1",
                }
            },
            "required": ["gene_symbol"],
        },
    }
]

# Simulate a database lookup (replace with a real API call in production)
def get_gene_info(gene_symbol: str) -> dict:
    db = {
        "BRCA1": {"chromosome": "17q21.31", "length_bp": 81189, "gc_content_pct": 42.1},
        "TP53":  {"chromosome": "17p13.1",  "length_bp": 19149, "gc_content_pct": 50.3},
    }
    return db.get(gene_symbol.upper(), {"error": "Gene not found"})

messages = [{"role": "user", "content": "What is the GC content of the BRCA1 gene?"}]

# Turn 1: model requests a tool
response = anthropic_client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=256,
    tools=tools,
    messages=messages,
)

print(f"Stop reason: {response.stop_reason}")
if response.stop_reason == "tool_use":
    tool_block = next(b for b in response.content if b.type == "tool_use")
    print(f"Model requested tool: {tool_block.name}({tool_block.input})")

    # Execute the tool
    tool_result = get_gene_info(**tool_block.input)

    # Turn 2: return tool result to the model
    messages += [
        {"role": "assistant", "content": response.content},
        {
            "role": "user",
            "content": [{
                "type": "tool_result",
                "tool_use_id": tool_block.id,
                "content": json.dumps(tool_result),
            }],
        },
    ]

    final = anthropic_client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=256,
        tools=tools,
        messages=messages,
    )
    print("\nWITH tools — final answer:")
    print(final.content[0].text)

In [ ]:
# ── OpenAI Chat Completions tool use ────────────────────────────────────────
oai_tools = [
    {
        "type": "function",
        "function": {
            "name": "get_gene_info",
            "description": "Retrieve GC content, chromosome, and length for a gene symbol.",
            "parameters": {
                "type": "object",
                "properties": {
                    "gene_symbol": {"type": "string", "description": "HGNC gene symbol"}
                },
                "required": ["gene_symbol"],
            },
        },
    }
]

oai_messages = [{"role": "user", "content": "What is the GC content of TP53?"}]

resp = openai_client.chat.completions.create(
    model="gpt-5.4-mini",
    tools=oai_tools,
    messages=oai_messages,
)

msg = resp.choices[0].message
if msg.tool_calls:
    call = msg.tool_calls[0]
    args = json.loads(call.function.arguments)
    print(f"Tool requested: {call.function.name}({args})")

    result = get_gene_info(**args)

    oai_messages += [
        msg,
        {"role": "tool", "tool_call_id": call.id, "content": json.dumps(result)},
    ]

    final = openai_client.chat.completions.create(
        model="gpt-5.4-mini", tools=oai_tools, messages=oai_messages
    )
    print("\nOpenAI WITH tools — final answer:")
    print(final.choices[0].message.content)

In [ ]:
# ── LangChain tool binding ───────────────────────────────────────────────────
from langchain_core.tools import tool

@tool
def get_gene_info_lc(gene_symbol: str) -> dict:
    """Retrieve GC content, chromosome, and length for a gene by its HGNC symbol."""
    db = {
        "BRCA1": {"chromosome": "17q21.31", "length_bp": 81189, "gc_content_pct": 42.1},
        "TP53":  {"chromosome": "17p13.1",  "length_bp": 19149, "gc_content_pct": 50.3},
    }
    return db.get(gene_symbol.upper(), {"error": "Gene not found"})

lc_model = ChatOpenAI(model="gpt-5.4-mini")
model_with_tools = lc_model.bind_tools([get_gene_info_lc])

ai_msg = model_with_tools.invoke("What chromosome is BRCA1 on?")
print("LangChain tool_calls on AIMessage:")
print(ai_msg.tool_calls)

In [ ]:
# ── LangGraph ReAct agent: automated tool-call loop ─────────────────────────
from langgraph.prebuilt import create_react_agent

agent = create_react_agent(
    model=ChatOpenAI(model="gpt-5.4-mini"),
    tools=[get_gene_info_lc],
)

result = agent.invoke({"messages": [{"role": "user", "content": "Compare the GC content of BRCA1 and TP53."}]})

print("LangGraph ReAct agent — final answer:")
print(result["messages"][-1].content)

---
## 6. Memory

LLMs are stateless by default — each API call has no knowledge of previous calls. Memory gives agents continuity.

| Memory type | How it works | Trade-off |
|---|---|---|
| **In-context (list)** | Append all messages to each call | Simple, but context fills up |
| **Summary memory** | Compress old turns before appending | Lossy, saves tokens |
| **LangGraph checkpointer** | Persist full graph state to storage | Scalable, supports multiple threads |

In [ ]:
# ── WITHOUT memory: each call is independent ────────────────────────────────
r1 = anthropic_client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=80,
    messages=[{"role": "user", "content": "My favorite organism is C. elegans."}],
)
print("Turn 1 (no memory):", r1.content[0].text)

r2 = anthropic_client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=80,
    messages=[{"role": "user", "content": "What is my favorite organism?"}],  # new call, no history
)
print("Turn 2 (no memory):", r2.content[0].text)

In [ ]:
# ── WITH in-context memory: accumulate messages manually ────────────────────
history = []

def chat_with_memory(user_input: str) -> str:
    history.append({"role": "user", "content": user_input})
    response = anthropic_client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=80,
        messages=history,
    )
    assistant_text = response.content[0].text
    history.append({"role": "assistant", "content": assistant_text})
    return assistant_text

print("Turn 1:", chat_with_memory("My favorite organism is C. elegans."))
print("Turn 2:", chat_with_memory("What is my favorite organism?"))
print("Turn 3:", chat_with_memory("What is it commonly used to study?"))

In [ ]:
# ── LangGraph checkpointer memory (persists across graph invocations) ────────
# thread_id isolates separate conversation sessions.
from langgraph.checkpoint.memory import MemorySaver
from langgraph.prebuilt import create_react_agent

checkpointer = MemorySaver()   # in-process store; swap for SqliteSaver for persistence

agent_with_memory = create_react_agent(
    model=ChatAnthropic(model="claude-sonnet-4-6"),
    tools=[],
    checkpointer=checkpointer,
)

config = {"configurable": {"thread_id": "session-001"}}  # unique per conversation

r1 = agent_with_memory.invoke(
    {"messages": [{"role": "user", "content": "My favorite organism is C. elegans."}]},
    config=config,
)
print("Turn 1:", r1["messages"][-1].content)

r2 = agent_with_memory.invoke(
    {"messages": [{"role": "user", "content": "What is my favorite organism?"}]},
    config=config,   # same thread_id — agent remembers
)
print("Turn 2:", r2["messages"][-1].content)

In [ ]:
# Different thread_id = fresh session (no shared memory)
r3 = agent_with_memory.invoke(
    {"messages": [{"role": "user", "content": "What is my favorite organism?"}]},
    config={"configurable": {"thread_id": "session-002"}},  # different thread
)
print("New thread (no memory):", r3["messages"][-1].content)

---
## 7. Streaming

Without streaming, the entire response arrives at once after the model finishes generating. With streaming, tokens arrive as they are generated — essential for interactive UIs and long outputs.

| Mode | Latency to first token | Best for |
|---|---|---|
| Non-streaming | Full generation time | Batch processing, short responses |
| Streaming | Milliseconds | Chat UIs, long summaries, notebooks |

In [ ]:
import time

prompt = "Explain the central dogma of molecular biology in 4 sentences."

# ── WITHOUT streaming: wait for the full response ───────────────────────────
t0 = time.time()
response = anthropic_client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=200,
    messages=[{"role": "user", "content": prompt}],
)
elapsed = time.time() - t0
print(f"Non-streaming — received full response after {elapsed:.1f}s:")
print(response.content[0].text)

In [ ]:
# ── Anthropic streaming — tokens print as they arrive ───────────────────────
print("Streaming (Anthropic) — tokens arrive incrementally:")
t0 = time.time()

with anthropic_client.messages.stream(
    model="claude-sonnet-4-6",
    max_tokens=200,
    messages=[{"role": "user", "content": prompt}],
) as stream:
    for text in stream.text_stream:
        print(text, end="", flush=True)

print(f"\n\nFirst token in ~0.5–1.5s, total: {time.time() - t0:.1f}s")

In [ ]:
# ── OpenAI streaming ─────────────────────────────────────────────────────────
print("Streaming (OpenAI Chat Completions):")

stream = openai_client.chat.completions.create(
    model="gpt-4o-mini",
    max_tokens=200,
    stream=True,
    messages=[{"role": "user", "content": prompt}],
)
for chunk in stream:
    delta = chunk.choices[0].delta.content
    if delta:
        print(delta, end="", flush=True)
print()

In [ ]:
# ── LangChain streaming ──────────────────────────────────────────────────────
print("Streaming (LangChain):")

model = ChatAnthropic(model="claude-sonnet-4-6", max_tokens=200)
for chunk in model.stream(prompt):
    print(chunk.content, end="", flush=True)
print()

In [ ]:
# ── LangGraph agent streaming — see each step as it happens ─────────────────
agent = create_react_agent(
    model=ChatOpenAI(model="gpt-5.4-mini"),
    tools=[get_gene_info_lc],
)

print("LangGraph streaming (stream_mode='values' shows state after each node):")
for state in agent.stream(
    {"messages": [{"role": "user", "content": "What is the GC content of BRCA1?"}]},
    stream_mode="values",
):
    last = state["messages"][-1]
    print(f"  [{type(last).__name__}]: {str(last.content)[:120]}")

---
## 8. Structured Output

By default, LLMs return free-form text. Structured output constrains the response to a defined schema — a Pydantic model — so downstream code can parse it reliably without regex or string manipulation.

**Without structured output**: free text — must be parsed manually, fragile.
**With structured output**: a typed Python object, validated automatically.

In [ ]:
from pydantic import BaseModel, Field
from typing import List, Literal

abstract = """
CRISPR-Cas9 was used to knock out the PTEN gene in HEK293 cells. Western blot confirmed
loss of PTEN protein. AKT phosphorylation increased 3.2-fold (p=0.001). Cell viability
was unaffected (p=0.45). These findings suggest PTEN loss activates the PI3K/AKT pathway
without affecting short-term cell survival.
"""

# ── WITHOUT structured output ────────────────────────────────────────────────
response = anthropic_client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=200,
    system="Extract key information from biomedical abstracts.",
    messages=[{"role": "user", "content": f"Extract the key findings from:\n{abstract}"}],
)
raw_text = response.content[0].text
print("WITHOUT structured output (raw text):")
print(raw_text)
print(f"\nType: {type(raw_text)} — must parse manually")

In [ ]:
# Define the output schema with Pydantic
class Finding(BaseModel):
    gene: str               = Field(description="Gene symbol mentioned")
    effect: str             = Field(description="Observed biological effect")
    p_value: float | None   = Field(description="p-value if reported", default=None)
    significant: bool       = Field(description="Whether the result is statistically significant")

class AbstractExtraction(BaseModel):
    method: str             = Field(description="Experimental method used")
    cell_line: str          = Field(description="Cell line or organism used")
    findings: List[Finding] = Field(description="List of key quantitative findings")
    conclusion: str         = Field(description="One-sentence conclusion")

In [ ]:
# ── WITH structured output: LangChain + with_structured_output() ─────────────
# Works with any LangChain-compatible model (Anthropic, OpenAI, etc.)
model = ChatAnthropic(model="claude-sonnet-4-6")
structured_model = model.with_structured_output(AbstractExtraction)

result = structured_model.invoke(
    f"Extract key information from this abstract:\n{abstract}"
)

print("WITH structured output (typed Python object):")
print(f"Type: {type(result).__name__}")
print(f"Method:     {result.method}")
print(f"Cell line:  {result.cell_line}")
print(f"Conclusion: {result.conclusion}")
print("\nFindings:")
for f in result.findings:
    print(f"  {f.gene}: {f.effect} | p={f.p_value} | significant={f.significant}")

In [ ]:
# ── OpenAI structured output via response_format ─────────────────────────────
# parse() validates against the Pydantic model automatically.
completion = openai_client.beta.chat.completions.parse(
    model="gpt-5.4-mini",
    messages=[
        {"role": "system", "content": "Extract key information from biomedical abstracts."},
        {"role": "user",   "content": f"Extract key information from:\n{abstract}"},
    ],
    response_format=AbstractExtraction,
)

oai_result = completion.choices[0].message.parsed
print("OpenAI structured output:")
print(f"Method: {oai_result.method}")
print(f"Conclusion: {oai_result.conclusion}")

In [ ]:
# ── Anthropic structured output via tool_use (recommended pattern) ───────────
# Anthropic doesn't have a native structured-output param, but tool_use
# with tool_choice forces a structured JSON response.
import json

extraction_tool = {
    "name": "extract_abstract",
    "description": "Extract structured information from a biomedical abstract.",
    "input_schema": AbstractExtraction.model_json_schema(),
}

response = anthropic_client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=512,
    tools=[extraction_tool],
    tool_choice={"type": "tool", "name": "extract_abstract"},  # force tool call
    messages=[{"role": "user", "content": f"Extract information from:\n{abstract}"}],
)

tool_input = response.content[0].input
parsed = AbstractExtraction(**tool_input)

print("Anthropic tool_use structured output:")
print(f"Method: {parsed.method}")
print(f"Conclusion: {parsed.conclusion}")
for f in parsed.findings:
    print(f"  {f.gene}: significant={f.significant}, p={f.p_value}")

In [ ]:
# ── Structured output for classification (simpler schema) ────────────────────
class SentimentResult(BaseModel):
    label: Literal["positive", "negative", "neutral"]
    confidence: float = Field(ge=0.0, le=1.0)
    reason: str

classifier = ChatOpenAI(model="gpt-5.4-mini").with_structured_output(SentimentResult)

texts = [
    "The CRISPR knockdown was highly efficient and reproducible.",
    "The western blot was overexposed and the bands are uninterpretable.",
    "We observed a 2-fold change in expression.",
]

print("Batch classification with structured output:\n")
for text in texts:
    result = classifier.invoke(f"Classify the sentiment of this scientific statement: '{text}'")
    print(f"  Text: {text[:60]}...")
    print(f"  → {result.label} (confidence={result.confidence:.2f}): {result.reason}\n")

---
## Summary

| Concept | Anthropic SDK | OpenAI SDK | LangChain / LangGraph |
|---|---|---|---|
| **Model** | `model="claude-sonnet-4-6"` | `model="gpt-5.4-mini"` | `ChatAnthropic(model=...)` |
| **API type** | `messages.create()` | `chat.completions.create()` or `responses.create()` | `.invoke()` / `.stream()` |
| **System prompt** | top-level `system=` param | `{"role": "system"}` message | `SystemMessage(...)` |
| **Human turn** | `{"role": "user"}` | `{"role": "user"}` | `HumanMessage(...)` |
| **Assistant turn** | `{"role": "assistant"}` | `{"role": "assistant"}` | `AIMessage(...)` |
| **Tools** | `tools=` + `tool_choice=` | `tools=` + `tool_calls` | `.bind_tools()` + `ToolNode` |
| **Memory** | Manual message list | `previous_response_id` (Responses API) | `MemorySaver` checkpointer |
| **Streaming** | `.stream()` context manager | `stream=True` + iterate chunks | `.stream()` |
| **Structured output** | Tool-use with forced tool call | `beta.chat.completions.parse()` | `.with_structured_output(Schema)` |

**Key takeaways**:
- Start simple: plain `messages.create()` / `chat.completions.create()` is enough for most tasks.
- Add a **system prompt** first — it is the highest-leverage configuration for controlling behavior.
- Add **tools** when the model needs access to data or actions beyond its training.
- Add **memory** (checkpointer) when conversations span multiple turns or sessions.
- Use **streaming** for any response longer than a sentence in an interactive setting.
- Use **structured output** whenever downstream code needs to parse the response.